# 04_4 — Entrenamiento LSTM Secuencial

Entrenamiento e inspección del modelo `price_sequence_lstm` usando exactamente el mismo contrato del GRU: secuencias de precios en grilla fija, rama estática snapshot-compatible, target residual `label_yes - price_yes`, calibración y split temporal agrupado.

Este notebook no cambia el dataset. Documenta el experimento LSTM ya entrenado desde terminal y permite reentrenarlo si se activa `RUN_TRAINING=True`.


In [ ]:
import sys
import json
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))

from src.config import load_config
from src.model.ts_dataset import TimeSeriesMarketDataset

cfg = load_config(ROOT / 'config' / 'config.yaml')
PROCESSED_TS = ROOT / cfg['ts_data']['processed_dir']
MODEL_DIR = ROOT / 'data' / 'models' / 'price_sequence_lstm'
RUN_TRAINING = False

PALETTE = ['#2A9D8F', '#264653', '#E76F51', '#8D99AE']


## 1. Dataset secuencial

El LSTM ve la misma entrada que el GRU: `price_yes_ffill`, `delta_price` y `observed_mask` durante 60 pasos de 12 horas.


In [ ]:
dataset = TimeSeriesMarketDataset.from_numpy_dir(str(PROCESSED_TS))
metadata = dataset.metadata

print(f"Samples        : {len(dataset):,}")
print(f"Sequence shape : {tuple(dataset.sequences.shape)}")
print(f"Positive rate  : {float(dataset.labels.mean()):.3f}")
print(f"Target         : {metadata.get('target_name')}")
print(f"Benchmark      : {metadata.get('benchmark')}")
print(f"Seq features   : {metadata.get('sequence_feature_names')}")


## 2. Entrenar o cargar artefactos

Por default se cargan los resultados guardados en `data/models/price_sequence_lstm/`. Para reentrenar desde el notebook cambia `RUN_TRAINING=True`.


In [ ]:
if RUN_TRAINING:
    cmd = [sys.executable, '-m', 'src.model.train', '--config', str(ROOT / 'config' / 'config.yaml'), '--only', 'lstm']
    print('Ejecutando:', ' '.join(cmd))
    subprocess.run(cmd, cwd=ROOT, check=True)
else:
    print('Usando artefactos existentes:', MODEL_DIR)

with open(MODEL_DIR / 'run_config.json') as f:
    run_config = json.load(f)
with open(MODEL_DIR / 'training_history.json') as f:
    history = json.load(f)
with open(MODEL_DIR / 'test_metrics.json') as f:
    metrics = json.load(f)

print('Modelo seleccionado:')
display(pd.Series(run_config['model']))
print()
print('Split:')
display(pd.Series(run_config['split']))


## 3. Búsqueda corta de hiperparámetros

Se probaron tres variantes pequeñas. La selección prioriza utilidad Top-K en validación, luego Brier y log-loss.


In [ ]:
search_df = pd.DataFrame([
    {
        'candidate': row['candidate'],
        'hidden_dim': row['model']['hidden_dim'],
        'num_layers': row['model']['num_layers'],
        'dropout': row['model']['dropout'],
        'best_epoch': row['best_epoch'],
        'best_val_loss': row['best_val_loss'],
        'val_brier': row['validation']['brier'],
        'val_log_loss': row['validation']['log_loss'],
        'val_topk_pnl': row['validation']['top_k_avg_realized_pnl'],
        'val_topk_hit_rate': row['validation']['top_k_hit_rate'],
    }
    for row in run_config.get('search_results', [])
])
display(search_df)

if not search_df.empty:
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    for ax, col, title in [
        (axes[0], 'val_brier', 'Brier val ↓'),
        (axes[1], 'val_log_loss', 'Log-loss val ↓'),
        (axes[2], 'val_topk_pnl', 'Top-K PnL val ↑'),
    ]:
        ax.bar(search_df['candidate'].astype(str), search_df[col], color=PALETTE[0], alpha=0.85)
        ax.axhline(0, color='black', lw=1)
        ax.set_title(title)
        ax.set_xlabel('Candidato')
    plt.tight_layout()


## 4. Curvas de entrenamiento

El LSTM hizo early stopping rápido. Eso es señal de que más epochs probablemente no arreglan el problema principal.


In [ ]:
epochs = np.arange(1, len(history.get('train_loss', [])) + 1)
best_epoch = history.get('best_epoch')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(epochs, history.get('train_loss', []), label='train', color=PALETTE[0])
axes[0].plot(epochs, history.get('val_loss', []), label='val', color=PALETTE[1])
if best_epoch:
    axes[0].axvline(best_epoch, color='gray', ls='--', label=f'best epoch={best_epoch}')
axes[0].set_title('Loss')
axes[0].legend()

axes[1].plot(epochs, history.get('val_brier', []), label='Brier', color=PALETTE[0])
axes[1].plot(epochs, history.get('val_log_loss', []), label='Log-loss', color=PALETTE[2])
axes[1].set_title('Métricas de validación')
axes[1].legend()
plt.tight_layout()


## 5. Test: LSTM vs mercado

La comparación crítica no es solo AUC: importa si mejora al precio de mercado y si rankea bien oportunidades con EV positivo.


In [ ]:
tm = metrics['test']
cal = tm['calibrated_metrics']
mkt = tm['market_baseline_metrics']
ev = tm['ev_metrics']
beaten = metrics.get('market_baseline_beaten', {})

summary = pd.DataFrame([
    {'metric': 'Brier ↓', 'LSTM': cal['brier'], 'Mercado': mkt['brier']},
    {'metric': 'Log-loss ↓', 'LSTM': cal['log_loss'], 'Mercado': mkt['log_loss']},
    {'metric': 'ECE ↓', 'LSTM': cal['ece'], 'Mercado': mkt['ece']},
    {'metric': 'ROC-AUC ↑', 'LSTM': cal['roc_auc'], 'Mercado': mkt['roc_auc']},
    {'metric': 'PR-AUC ↑', 'LSTM': cal['pr_auc'], 'Mercado': mkt['pr_auc']},
])
display(summary)
print('Market baseline beaten:', beaten)
print(f"Top-K avg realized PnL: {ev['top_k_avg_realized_pnl']:.4f}")
print(f"Top-K hit rate        : {ev['top_k_hit_rate']:.3f}")
print(f"Top-K predicted EV    : {ev['top_k_avg_predicted_ev']:.4f}")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
prob_metrics = ['brier', 'log_loss', 'ece']
x = np.arange(len(prob_metrics))
w = 0.35
axes[0].bar(x - w/2, [cal[m] for m in prob_metrics], w, label='LSTM', color=PALETTE[0])
axes[0].bar(x + w/2, [mkt[m] for m in prob_metrics], w, label='Mercado', color=PALETTE[3])
axes[0].set_xticks(x)
axes[0].set_xticklabels(['Brier', 'Log-loss', 'ECE'])
axes[0].set_title('Probabilidad calibrada')
axes[0].legend()

axes[1].bar(['Top-K PnL', 'Top-K hit'], [ev['top_k_avg_realized_pnl'], ev['top_k_hit_rate']], color=[PALETTE[2], PALETTE[0]])
axes[1].axhline(0, color='black', lw=1)
axes[1].set_title('Utilidad Top-K')
plt.tight_layout()


## 6. Por horizonte

El LSTM queda detrás del mercado en los tres buckets relevantes; el problema es más fuerte en utilidad Top-K.


In [ ]:
rows = []
for bucket, payload in tm.get('by_horizon', {}).items():
    rows.append({
        'bucket': bucket,
        'count': payload.get('count', 0),
        'lstm_brier': payload.get('probability_metrics', {}).get('brier'),
        'market_brier': payload.get('market_baseline_metrics', {}).get('brier'),
        'topk_pnl': payload.get('ev_metrics', {}).get('top_k_avg_realized_pnl'),
        'topk_hit_rate': payload.get('ev_metrics', {}).get('top_k_hit_rate'),
    })
horizon_df = pd.DataFrame(rows)
display(horizon_df)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
x = np.arange(len(horizon_df))
w = 0.35
axes[0].bar(x - w/2, horizon_df['lstm_brier'], w, label='LSTM', color=PALETTE[0])
axes[0].bar(x + w/2, horizon_df['market_brier'], w, label='Mercado', color=PALETTE[3])
axes[0].set_xticks(x)
axes[0].set_xticklabels(horizon_df['bucket'], rotation=20, ha='right')
axes[0].set_title('Brier por horizonte')
axes[0].legend()

axes[1].bar(horizon_df['bucket'], horizon_df['topk_pnl'], color=PALETTE[2])
axes[1].axhline(0, color='black', lw=1)
axes[1].set_xticklabels(horizon_df['bucket'], rotation=20, ha='right')
axes[1].set_title('Top-K PnL por horizonte')
plt.tight_layout()


## 7. Lectura final

El LSTM sí aprende señal de ranking general, pero no supera al mercado ni al HistBoost/CatBoost en calibración o utilidad económica. Para la presentación conviene usarlo como experimento negativo: más complejidad secuencial no necesariamente mejora una tarea donde `price_yes` ya resume mucha información.
